# Temporal Analysis - CICIDS2017 (PySpark Edition)

This notebook analyzes network traffic patterns across the 8 capture scenarios using **PySpark** - Apache Spark's Python API for distributed computing.

**Contents:**
1. Scenario Mapping & Overview
2. Traffic Volume by Day/Scenario
3. Attack Distribution by Scenario
4. Benign Traffic Stability Across Days
5. Temporal Train/Test Split Experiment
6. Feature Drift Analysis

**PySpark Syntax Highlights:**
- `spark.read.parquet()` to load data
- `df.select()`, `df.filter()`, `df.groupBy()` similar to SQL
- `F.col()` for column references
- `F.when().otherwise()` for conditionals
- `.collect()` or `.toPandas()` to bring data to driver

In [1]:
# Install PySpark if needed
# !pip install pyspark pyarrow

In [2]:
# Initialize Spark Session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Create a local Spark session
spark = SparkSession.builder \
    .appName("CICIDS2017-Temporal-Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Set log level to reduce output noise
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print("Spark Session created successfully!")

Error: LinkageError occurred while loading main class org.apache.spark.launcher.Main
	java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 55.0
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# Additional imports for visualization and ML
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support, average_precision_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("Libraries loaded successfully")

In [ ]:
# Load processed data with PySpark
df = spark.read.parquet("../data/processed/cicids2017_clean.parquet")

# Cache the DataFrame for faster operations
df.cache()

print(f"Dataset shape: ({df.count()}, {len(df.columns)})")
print(f"\nMeta_source values (capture scenarios):")
df.select('Meta_source').distinct().show(truncate=False)

In [ ]:
# View schema - PySpark's equivalent to pandas dtypes
df.printSchema()

In [ ]:
# Preview data - PySpark style
df.show(5, truncate=True)

---
## 1. Scenario Mapping & Overview

The CICIDS2017 dataset was captured over 5 days, with specific attack types executed during each capture session.

In [ ]:
# Define scenario mapping as a Python dict (will display as reference)
SCENARIO_INFO = {
    'Monday-WorkingHours': {
        'day': 'Monday',
        'attacks': ['BENIGN only'],
        'description': 'Normal benign traffic baseline'
    },
    'Tuesday-WorkingHours': {
        'day': 'Tuesday', 
        'attacks': ['FTP-Patator', 'SSH-Patator'],
        'description': 'Brute force attacks on FTP and SSH'
    },
    'Wednesday-workingHours': {
        'day': 'Wednesday',
        'attacks': ['DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye', 'Heartbleed'],
        'description': 'DoS attacks and Heartbleed vulnerability exploit'
    },
    'Thursday-WorkingHours-Morning-WebAttacks': {
        'day': 'Thursday AM',
        'attacks': ['Web Attack - Brute Force', 'Web Attack - XSS', 'Web Attack - Sql Injection'],
        'description': 'Web application attacks'
    },
    'Thursday-WorkingHours-Afternoon-Infilteration': {
        'day': 'Thursday PM',
        'attacks': ['Infiltration'],
        'description': 'Infiltration and lateral movement'
    },
    'Friday-WorkingHours-Morning': {
        'day': 'Friday AM',
        'attacks': ['Bot'],
        'description': 'Botnet command and control traffic'
    },
    'Friday-WorkingHours-Afternoon-PortScan': {
        'day': 'Friday PM (1)',
        'attacks': ['PortScan'],
        'description': 'Network reconnaissance via port scanning'
    },
    'Friday-WorkingHours-Afternoon-DDos': {
        'day': 'Friday PM (2)',
        'attacks': ['DDoS'],
        'description': 'Distributed denial of service attack'
    }
}

# Display scenario table
print("CICIDS2017 Capture Scenarios:")
print("=" * 100)
for scenario, info in SCENARIO_INFO.items():
    print(f"{info['day']:<15} | {scenario:<50} | {', '.join(info['attacks'])}")

In [ ]:
# Extract day of week from Meta_source using PySpark F.when()
# This is PySpark's equivalent of CASE WHEN in SQL
df = df.withColumn(
    'Day',
    F.when(F.col('Meta_source').contains('Monday'), 'Monday')
    .when(F.col('Meta_source').contains('Tuesday'), 'Tuesday')
    .when(F.col('Meta_source').contains('Wednesday'), 'Wednesday')
    .when(F.col('Meta_source').contains('Thursday'), 'Thursday')
    .when(F.col('Meta_source').contains('Friday'), 'Friday')
    .otherwise('Unknown')
)

# Add Is_Attack column (binary)
df = df.withColumn(
    'Is_Attack',
    F.when(F.col('Label') != 'BENIGN', 1).otherwise(0)
)

# Day order for plotting
DAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

print("Day distribution:")
df.groupBy('Day').count().orderBy('Day').show()

---
## 2. Traffic Volume by Day/Scenario

Understanding the volume and composition of traffic across capture sessions.

In [ ]:
# Traffic volume by scenario using PySpark groupBy
scenario_counts = (
    df
    .groupBy('Meta_source')
    .count()
    .orderBy(F.desc('count'))
)

# Collect to driver for visualization (converts to list of Row objects)
scenario_counts_list = scenario_counts.collect()

print("Traffic Volume by Scenario:")
scenario_counts.show(truncate=False)

In [ ]:
# Plot traffic volume by scenario
fig, ax = plt.subplots(figsize=(14, 6))

scenarios = [row['Meta_source'] for row in scenario_counts_list]
counts = [row['count'] for row in scenario_counts_list]

bars = ax.bar(range(len(scenarios)), counts, color='steelblue', edgecolor='black')
ax.set_title('Total Traffic Volume by Capture Scenario', fontsize=14, fontweight='bold')
ax.set_xlabel('Capture Scenario')
ax.set_ylabel('Number of Flows')
ax.set_xticks(range(len(scenarios)))
ax.set_xticklabels(scenarios, rotation=45, ha='right')

# Add count labels
for i, v in enumerate(counts):
    ax.text(i, v + 5000, f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/traffic_volume_by_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Traffic volume by day (aggregated) using PySpark
day_counts = (
    df
    .groupBy('Day')
    .agg(
        F.count('*').alias('Total'),
        F.sum('Is_Attack').alias('Attack_Count')
    )
    .withColumn('Benign_Count', F.col('Total') - F.col('Attack_Count'))
)

# Convert to pandas for easier plotting
day_counts_pd = day_counts.toPandas()
day_counts_pd = day_counts_pd.set_index('Day').reindex(DAY_ORDER)

print("Traffic by Day:")
print(day_counts_pd)

In [ ]:
# Plot traffic by day
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total traffic by day
ax1 = axes[0]
ax1.bar(DAY_ORDER, day_counts_pd['Total'], color='steelblue', edgecolor='black')
ax1.set_title('Total Traffic by Day of Week', fontsize=12, fontweight='bold')
ax1.set_xlabel('Day')
ax1.set_ylabel('Number of Flows')

# Stacked: benign vs attack
ax2 = axes[1]
ax2.bar(DAY_ORDER, day_counts_pd['Benign_Count'], label='Benign', color='green', alpha=0.7)
ax2.bar(DAY_ORDER, day_counts_pd['Attack_Count'], bottom=day_counts_pd['Benign_Count'], 
        label='Attack', color='red', alpha=0.7)
ax2.set_title('Traffic Composition by Day (Benign vs Attack)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Day')
ax2.set_ylabel('Number of Flows')
ax2.legend()

plt.tight_layout()
plt.savefig('../reports/figures/traffic_by_day.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Attack percentage by scenario using PySpark
scenario_attack_pct = (
    df
    .groupBy('Meta_source')
    .agg(
        F.count('*').alias('Total'),
        F.sum('Is_Attack').alias('Attack_Count')
    )
    .withColumn('Attack_Pct', (F.col('Attack_Count') / F.col('Total') * 100))
    .orderBy('Meta_source')
)

# Convert to pandas
attack_pct_pd = scenario_attack_pct.toPandas()

print("Attack Percentage by Scenario:")
print(attack_pct_pd[['Meta_source', 'Attack_Pct']].round(2))

In [ ]:
# Plot attack percentage
fig, ax = plt.subplots(figsize=(14, 5))

colors = ['red' if pct > 50 else ('orange' if pct > 10 else 'green') 
          for pct in attack_pct_pd['Attack_Pct']]

ax.bar(range(len(attack_pct_pd)), attack_pct_pd['Attack_Pct'], color=colors, edgecolor='black')
ax.set_title('Attack Traffic Percentage by Scenario', fontsize=14, fontweight='bold')
ax.set_xlabel('Scenario')
ax.set_ylabel('Attack Percentage (%)')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.set_xticks(range(len(attack_pct_pd)))
ax.set_xticklabels(attack_pct_pd['Meta_source'], rotation=45, ha='right')
ax.legend()

# Add percentage labels
for i, v in enumerate(attack_pct_pd['Attack_Pct']):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/attack_percentage_by_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Attack Distribution by Scenario

Which attack types appear in which scenarios? Verifying against expected attack schedule.

In [ ]:
# Cross-tabulation of attacks by scenario using PySpark
# This is like pandas crosstab - we use groupBy and pivot
attack_scenario_counts = (
    df
    .groupBy('Label')
    .pivot('Meta_source')
    .count()
    .fillna(0)
    .orderBy('Label')
)

print("Attack Types by Scenario (counts):")
attack_scenario_counts.show(truncate=False)

In [ ]:
# Convert to pandas for heatmap visualization
crosstab_pd = attack_scenario_counts.toPandas().set_index('Label')

# Normalize by row (attack type)
crosstab_norm = crosstab_pd.div(crosstab_pd.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(crosstab_norm, annot=True, fmt='.1f', cmap='YlOrRd', 
            cbar_kws={'label': '% of Attack Type'}, ax=ax)
ax.set_title('Attack Type Distribution Across Scenarios\n(% of each attack type per scenario)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Capture Scenario')
ax.set_ylabel('Attack Type')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/attack_scenario_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Verify attacks match expected scenarios using PySpark
print("Verification: Attack Types per Scenario")
print("=" * 80)

# Get distinct labels per scenario
labels_per_scenario = (
    df
    .filter(F.col('Label') != 'BENIGN')
    .groupBy('Meta_source')
    .agg(F.collect_set('Label').alias('Attacks'))
    .collect()
)

for row in labels_per_scenario:
    scenario = row['Meta_source']
    actual_attacks = row['Attacks']
    
    if scenario in SCENARIO_INFO:
        expected = SCENARIO_INFO[scenario]['attacks']
        if 'BENIGN only' in expected:
            expected = []
    else:
        expected = ['Unknown']
    
    print(f"\n{scenario}:")
    print(f"  Expected: {', '.join(expected) if expected else 'Benign only'}")
    print(f"  Actual:   {', '.join(actual_attacks) if actual_attacks else 'Benign only'}")

---
## 4. Benign Traffic Stability Across Days

Is benign traffic consistent across days, or does it vary?

In [ ]:
# Get benign traffic only using PySpark filter
df_benign = df.filter(F.col('Label') == 'BENIGN')

# Cache for repeated use
df_benign.cache()

benign_count = df_benign.count()
print(f"Benign traffic: {benign_count:,} flows")

print(f"\nBenign by day:")
df_benign.groupBy('Day').count().orderBy('Day').show()

In [ ]:
# Compare key features of benign traffic across days using PySpark
key_features = ['Flow Duration', 'Flow Bytes/s', 'Total Fwd Packets', 
                'Packet Length Mean', 'Flow IAT Mean', 'Destination Port']

# Calculate mean and std for each feature by day
# Build aggregation expressions dynamically
agg_exprs = []
for feat in key_features:
    # Use backticks for column names with special characters
    agg_exprs.append(F.mean(F.col(f"`{feat}`")).alias(f"{feat}_mean"))
    agg_exprs.append(F.stddev(F.col(f"`{feat}`")).alias(f"{feat}_std"))

benign_stats_by_day = df_benign.groupBy('Day').agg(*agg_exprs)

print("Benign Traffic Statistics by Day:")
benign_stats_by_day.show(truncate=False)

In [ ]:
# Sample benign data for visualization
# PySpark sample() method
df_benign_sample = df_benign.sample(fraction=0.1, seed=42)

# Convert to pandas for seaborn plots
df_benign_sample_pd = df_benign_sample.select(
    'Day', 'Flow Duration', 'Flow Bytes/s', 'Total Fwd Packets',
    'Packet Length Mean', 'Flow IAT Mean', 'Destination Port'
).toPandas()

print(f"Sample size: {len(df_benign_sample_pd):,}")

In [ ]:
# Box plots of benign features by day
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    ax = axes[idx]
    
    # Filter positive values for log scale
    data = df_benign_sample_pd[df_benign_sample_pd[feature] > 0]
    
    sns.boxplot(data=data, x='Day', y=feature, order=DAY_ORDER, ax=ax, showfliers=False)
    ax.set_title(f'Benign: {feature}', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    
    # Use log scale for highly skewed features
    if feature in ['Flow Duration', 'Flow Bytes/s']:
        ax.set_yscale('log')

plt.suptitle('Benign Traffic Feature Distributions by Day', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/benign_features_by_day.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical test for feature stability
# We need to collect data per day for scipy stats
print("Feature Stability Test (Kruskal-Wallis H-test):")
print("=" * 70)
print(f"{'Feature':<30} {'H-statistic':<15} {'p-value':<15} {'Stable?':<10}")
print("-" * 70)

stability_results = []

for feature in key_features:
    groups = []
    for day in DAY_ORDER:
        # Sample per day using PySpark and collect
        day_data = (
            df_benign
            .filter(F.col('Day') == day)
            .select(F.col(f"`{feature}`"))
            .sample(fraction=0.05, seed=42)
            .toPandas()
        )
        groups.append(day_data[feature].dropna().values)
    
    # Filter out empty groups
    groups = [g for g in groups if len(g) > 0]
    
    if len(groups) >= 2:
        stat, p_value = stats.kruskal(*groups)
        is_stable = 'Yes' if p_value > 0.05 else 'No'
    else:
        stat, p_value = np.nan, np.nan
        is_stable = 'N/A'
    
    stability_results.append({
        'Feature': feature,
        'H-statistic': stat,
        'p-value': p_value,
        'Stable': is_stable
    })
    
    print(f"{feature:<30} {stat:<15.2f} {p_value:<15.4e} {is_stable:<10}")

print("\nNote: 'Stable' means p > 0.05 (no significant difference across days)")

---
## 5. Temporal Train/Test Split Experiment

How well do models generalize when trained on early days and tested on later days?

In [ ]:
# Get numeric feature columns
exclude_cols = ['Label', 'Meta_source', 'Day', 'Is_Attack']

# Get numeric columns from schema
feature_cols = [field.name for field in df.schema.fields 
                if field.name not in exclude_cols 
                and field.dataType in [DoubleType(), FloatType(), IntegerType(), LongType()]]

print(f"Using {len(feature_cols)} features")

In [ ]:
# Experiment 1: Train on Mon-Thu, Test on Friday
print("Experiment 1: Train on Monday-Thursday, Test on Friday")
print("=" * 60)

# Filter using PySpark
df_train = df.filter(F.col('Day').isin(['Monday', 'Tuesday', 'Wednesday', 'Thursday']))
df_test = df.filter(F.col('Day') == 'Friday')

print(f"Train set (Mon-Thu): {df_train.count():,} samples")
train_stats = df_train.groupBy('Is_Attack').count().collect()
for row in train_stats:
    label = 'Benign' if row['Is_Attack'] == 0 else 'Attack'
    print(f"  - {label}: {row['count']:,}")

print(f"\nTest set (Friday): {df_test.count():,} samples")
test_stats = df_test.groupBy('Is_Attack').count().collect()
for row in test_stats:
    label = 'Benign' if row['Is_Attack'] == 0 else 'Attack'
    print(f"  - {label}: {row['count']:,}")

In [ ]:
# Sample training data for faster training (collect to driver)
# Note: For large datasets, use Spark MLlib instead
train_sample = df_train.sample(fraction=0.1, seed=42)
train_pd = train_sample.select(feature_cols + ['Is_Attack']).toPandas()

# Prepare arrays
X_train = train_pd[feature_cols].values
y_train = train_pd['Is_Attack'].values

# Handle inf values
X_train = np.nan_to_num(X_train, nan=0, posinf=0, neginf=0)

print(f"Training sample shape: {X_train.shape}")

In [ ]:
# Prepare test data
test_pd = df_test.select(feature_cols + ['Is_Attack']).toPandas()
X_test_temporal = test_pd[feature_cols].values
y_test_temporal = test_pd['Is_Attack'].values

X_test_temporal = np.nan_to_num(X_test_temporal, nan=0, posinf=0, neginf=0)

print(f"Test set shape: {X_test_temporal.shape}")

In [ ]:
# Scale and train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test_temporal)

print("Training Logistic Regression (temporal split)...")
lr_temporal = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_temporal.fit(X_train_scaled, y_train)

# Predict
y_pred_temporal = lr_temporal.predict(X_test_scaled)
y_proba_temporal = lr_temporal.predict_proba(X_test_scaled)[:, 1]

print("\nTemporal Split Results (Train: Mon-Thu, Test: Friday):")
print(classification_report(y_test_temporal, y_pred_temporal, target_names=['Benign', 'Attack']))

In [ ]:
# Baseline: Random split (for comparison)
print("\nBaseline: Random 80/20 Split")
print("=" * 60)

# Sample from full dataset using PySpark
full_sample = df.sample(fraction=0.1, seed=42)
full_pd = full_sample.select(feature_cols + ['Is_Attack']).toPandas()

X_full = np.nan_to_num(full_pd[feature_cols].values, nan=0, posinf=0, neginf=0)
y_full = full_pd['Is_Attack'].values

from sklearn.model_selection import train_test_split
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=42
)

# Scale and train
scaler_random = StandardScaler()
X_train_random_scaled = scaler_random.fit_transform(X_train_random)
X_test_random_scaled = scaler_random.transform(X_test_random)

print("Training Logistic Regression (random split)...")
lr_random = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_random.fit(X_train_random_scaled, y_train_random)

y_pred_random = lr_random.predict(X_test_random_scaled)
y_proba_random = lr_random.predict_proba(X_test_random_scaled)[:, 1]

print("\nRandom Split Results:")
print(classification_report(y_test_random, y_pred_random, target_names=['Benign', 'Attack']))

In [ ]:
# Compare PR-AUC
pr_auc_temporal = average_precision_score(y_test_temporal, y_proba_temporal)
pr_auc_random = average_precision_score(y_test_random, y_proba_random)

# Calculate precision and recall
prec_t, rec_t, f1_t, _ = precision_recall_fscore_support(y_test_temporal, y_pred_temporal, average='binary')
prec_r, rec_r, f1_r, _ = precision_recall_fscore_support(y_test_random, y_pred_random, average='binary')

import pandas as pd
comparison = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1-Score', 'PR-AUC'],
    'Temporal Split (Mon-Thu -> Fri)': [prec_t, rec_t, f1_t, pr_auc_temporal],
    'Random Split (80/20)': [prec_r, rec_r, f1_r, pr_auc_random]
})
comparison['Difference'] = comparison['Temporal Split (Mon-Thu -> Fri)'] - comparison['Random Split (80/20)']

print("\nComparison: Temporal vs Random Split")
print("=" * 60)
print(comparison.round(4).to_string(index=False))

print("\nInterpretation:")
if pr_auc_temporal < pr_auc_random * 0.95:
    print("  - Temporal split shows degraded performance (>5% drop)")
    print("  - This suggests temporal drift or attack type differences")
else:
    print("  - Model generalizes well across time")
    print("  - Attack patterns are consistent Mon-Thu vs Friday")

In [ ]:
# Plot PR curves
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(10, 8))

# Temporal split PR curve
precision_t, recall_t, _ = precision_recall_curve(y_test_temporal, y_proba_temporal)
ax.plot(recall_t, precision_t, label=f'Temporal Split (PR-AUC={pr_auc_temporal:.3f})', linewidth=2)

# Random split PR curve
precision_r, recall_r, _ = precision_recall_curve(y_test_random, y_proba_random)
ax.plot(recall_r, precision_r, label=f'Random Split (PR-AUC={pr_auc_random:.3f})', linewidth=2)

# Baseline
baseline = y_test_temporal.mean()
ax.axhline(y=baseline, color='gray', linestyle='--', label=f'Baseline ({baseline:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve: Temporal vs Random Split', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/pr_curve_temporal_vs_random.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Leave-one-day-out cross-validation
print("\nLeave-One-Day-Out Cross-Validation")
print("=" * 60)

lodo_results = []

for test_day in DAY_ORDER:
    # Train on all other days using PySpark filters
    df_train_lodo = df.filter(F.col('Day') != test_day).sample(fraction=0.1, seed=42)
    df_test_lodo = df.filter(F.col('Day') == test_day)
    
    # Collect to pandas
    train_lodo_pd = df_train_lodo.select(feature_cols + ['Is_Attack']).toPandas()
    test_lodo_pd = df_test_lodo.select(feature_cols + ['Is_Attack']).toPandas()
    
    X_train_lodo = np.nan_to_num(train_lodo_pd[feature_cols].values, nan=0, posinf=0, neginf=0)
    y_train_lodo = train_lodo_pd['Is_Attack'].values
    X_test_lodo = np.nan_to_num(test_lodo_pd[feature_cols].values, nan=0, posinf=0, neginf=0)
    y_test_lodo = test_lodo_pd['Is_Attack'].values
    
    # Scale
    scaler_lodo = StandardScaler()
    X_train_lodo_scaled = scaler_lodo.fit_transform(X_train_lodo)
    X_test_lodo_scaled = scaler_lodo.transform(X_test_lodo)
    
    # Train
    lr_lodo = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
    lr_lodo.fit(X_train_lodo_scaled, y_train_lodo)
    
    # Predict
    y_pred_lodo = lr_lodo.predict(X_test_lodo_scaled)
    y_proba_lodo = lr_lodo.predict_proba(X_test_lodo_scaled)[:, 1]
    
    # Metrics
    prec, rec, f1, _ = precision_recall_fscore_support(y_test_lodo, y_pred_lodo, average='binary', zero_division=0)
    pr_auc = average_precision_score(y_test_lodo, y_proba_lodo) if y_test_lodo.sum() > 0 else 0
    
    lodo_results.append({
        'Test Day': test_day,
        'Test Size': len(y_test_lodo),
        'Attack %': y_test_lodo.mean() * 100,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'PR-AUC': pr_auc
    })

lodo_df = pd.DataFrame(lodo_results)
print("\nLeave-One-Day-Out Results:")
print(lodo_df.round(3).to_string(index=False))

In [ ]:
# Visualize LODO results
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(DAY_ORDER))
width = 0.25

ax.bar(x - width, lodo_df['Precision'], width, label='Precision', color='steelblue')
ax.bar(x, lodo_df['Recall'], width, label='Recall', color='coral')
ax.bar(x + width, lodo_df['F1'], width, label='F1-Score', color='seagreen')

ax.set_xlabel('Test Day (trained on all other days)')
ax.set_ylabel('Score')
ax.set_title('Leave-One-Day-Out Cross-Validation Results', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(DAY_ORDER)
ax.legend()
ax.set_ylim(0, 1.1)

# Add attack % annotation
for i, row in lodo_df.iterrows():
    ax.annotate(f"{row['Attack %']:.0f}% atk", (i, 1.02), ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('../reports/figures/lodo_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Feature Drift Analysis

Identifying which features are stable across days vs which show significant drift.

In [ ]:
# Compute feature stability using PySpark aggregations
# Calculate mean per day for benign traffic

feature_drift_scores = []

for feature in feature_cols[:30]:  # Analyze first 30 features for speed
    # Get mean per day using PySpark
    day_means = (
        df_benign
        .groupBy('Day')
        .agg(F.mean(F.col(f"`{feature}`")).alias('mean'))
        .toPandas()
    )
    
    # Coefficient of variation
    cv = day_means['mean'].std() / (day_means['mean'].mean() + 1e-10)
    
    feature_drift_scores.append({
        'Feature': feature,
        'CV': cv
    })

drift_df = pd.DataFrame(feature_drift_scores).sort_values('CV', ascending=False)
print("Feature variability (Coefficient of Variation):")
print(drift_df.head(15).round(4))

In [ ]:
# Visualize feature stability
fig, ax = plt.subplots(figsize=(12, 8))

# Top 20 most variable
top_variable = drift_df.head(20)

ax.barh(range(len(top_variable)), top_variable['CV'], color='steelblue', alpha=0.7)
ax.set_yticks(range(len(top_variable)))
ax.set_yticklabels(top_variable['Feature'])
ax.set_xlabel('Coefficient of Variation (higher = more variable)')
ax.set_title('Top 20 Most Variable Features Across Days', fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/figures/feature_drift_top20.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Identify stable vs unstable features
cv_threshold = drift_df['CV'].median()

stable_features = drift_df[drift_df['CV'] <= cv_threshold]['Feature'].tolist()
unstable_features = drift_df[drift_df['CV'] > cv_threshold]['Feature'].tolist()

print(f"\nFeature Stability Summary:")
print(f"  CV threshold (median): {cv_threshold:.4f}")
print(f"  Stable features (CV <= threshold): {len(stable_features)}")
print(f"  Unstable features (CV > threshold): {len(unstable_features)}")

print(f"\nStable features recommended for temporal-robust models:")
for i, feat in enumerate(stable_features[:10]):
    print(f"  {i+1}. {feat}")
if len(stable_features) > 10:
    print(f"  ... and {len(stable_features) - 10} more")

---
## Summary & Key Findings

In [ ]:
print("TEMPORAL ANALYSIS - KEY FINDINGS")
print("=" * 80)

# Get total count
total_count = df.count()

print("\n1. SCENARIO OVERVIEW:")
print(f"   - 8 capture scenarios across 5 days")
print(f"   - Total flows: {total_count:,}")
print(f"   - Monday has benign-only traffic (baseline)")

print("\n2. TRAFFIC COMPOSITION:")
for _, row in day_counts_pd.iterrows():
    atk_pct = row['Attack_Count'] / row['Total'] * 100
    print(f"   - {row.name}: {int(row['Total']):,} flows ({atk_pct:.1f}% attacks)")

print("\n3. TEMPORAL GENERALIZATION:")
print(f"   - Temporal split PR-AUC: {pr_auc_temporal:.3f}")
print(f"   - Random split PR-AUC: {pr_auc_random:.3f}")
print(f"   - Difference: {pr_auc_temporal - pr_auc_random:.3f}")

print("\n4. FEATURE STABILITY:")
print(f"   - Stable features: {len(stable_features)}/{len(drift_df)}")
print(f"   - Unstable features: {len(unstable_features)}/{len(drift_df)}")

print("\n5. RECOMMENDATIONS:")
print("   - Use stable features for production models")
print("   - Monitor for drift on Friday-specific attack types")
print("   - Consider temporal train/test splits for realistic evaluation")

In [ ]:
# Save analysis artifacts
import json
import os

os.makedirs('../reports/figures', exist_ok=True)

# Save stable features list
with open('../reports/stable_features.json', 'w') as f:
    json.dump(stable_features, f, indent=2)

# Save LODO results
lodo_df.to_csv('../reports/lodo_cv_results.csv', index=False)

# Save feature drift scores
drift_df.to_csv('../reports/feature_drift_scores.csv', index=False)

print("\nArtifacts saved to reports/:")
print("  - stable_features.json")
print("  - lodo_cv_results.csv")
print("  - feature_drift_scores.csv")
print("  - figures/*.png")

In [ ]:
# Stop Spark session when done
# spark.stop()
# Uncomment above to stop Spark

---
## PySpark Cheat Sheet

| Pandas | PySpark |
|--------|--------|
| `df['col']` | `df.select('col')` |
| `df[['a', 'b']]` | `df.select('a', 'b')` |
| `df[df['col'] > 5]` | `df.filter(F.col('col') > 5)` |
| `df['new'] = val` | `df.withColumn('new', F.lit(val))` |
| `df.groupby('col').agg()` | `df.groupBy('col').agg()` |
| `df.shape[0]` | `df.count()` |
| `len(df.columns)` | `len(df.columns)` |
| `df.values` | `df.toPandas().values` or `df.collect()` |
| `df.to_csv()` | `df.write.csv()` or `df.toPandas().to_csv()` |
| `pd.read_parquet()` | `spark.read.parquet()` |
| `df.apply(func)` | `df.withColumn('x', F.udf(func)(F.col('x')))` |
| `np.where(cond, a, b)` | `F.when(cond, a).otherwise(b)` |
| `df.dtypes` | `df.dtypes` or `df.printSchema()` |
| `df.head(5)` | `df.show(5)` or `df.limit(5).toPandas()` |
| `df.describe()` | `df.describe().show()` |
| `pd.crosstab()` | `df.groupBy().pivot().count()` |